### Practica
#### Nombre: Patricio Quishpe
#### Modelo de Predicción de Precios de Diamantes


## 1. Importación de librerías y carga del dataset

In [25]:
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Cargar el dataset 'diamonds' desde seaborn
diamonds = sns.load_dataset('diamonds')

# Mostrar las primeras filas del dataset
print("Primeras filas del dataset:")
print(diamonds.head())

Primeras filas del dataset:
   carat      cut color clarity  depth  table  price     x     y     z
0   0.23    Ideal     E     SI2   61.5   55.0    326  3.95  3.98  2.43
1   0.21  Premium     E     SI1   59.8   61.0    326  3.89  3.84  2.31
2   0.23     Good     E     VS1   56.9   65.0    327  4.05  4.07  2.31
3   0.29  Premium     I     VS2   62.4   58.0    334  4.20  4.23  2.63
4   0.31     Good     J     SI2   63.3   58.0    335  4.34  4.35  2.75


## 2. Exploración del Dataset

In [26]:
# Verificar valores nulos
print("\n=== VALORES NULOS ===")
print(diamonds.isnull().sum())
print(f"\nTotal de valores nulos: {diamonds.isnull().sum().sum()}")


=== VALORES NULOS ===
carat      0
cut        0
color      0
clarity    0
depth      0
table      0
price      0
x          0
y          0
z          0
dtype: int64

Total de valores nulos: 0


In [27]:
# Información general del dataset
print("\n=== INFORMACIÓN DEL DATASET ===")
print(diamonds.info())


=== INFORMACIÓN DEL DATASET ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53940 entries, 0 to 53939
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype   
---  ------   --------------  -----   
 0   carat    53940 non-null  float64 
 1   cut      53940 non-null  category
 2   color    53940 non-null  category
 3   clarity  53940 non-null  category
 4   depth    53940 non-null  float64 
 5   table    53940 non-null  float64 
 6   price    53940 non-null  int64   
 7   x        53940 non-null  float64 
 8   y        53940 non-null  float64 
 9   z        53940 non-null  float64 
dtypes: category(3), float64(6), int64(1)
memory usage: 3.0 MB
None


In [28]:
# Verificar categorías de las variables categóricas
print("\n=== CATEGORÍAS DE VARIABLES CATEGÓRICAS ===")
categorical_columns = diamonds.select_dtypes(include=['object', 'category']).columns

for col in categorical_columns:
    print(f"\n{col.upper()}:")
    print(diamonds[col].value_counts())


=== CATEGORÍAS DE VARIABLES CATEGÓRICAS ===

CUT:
cut
Ideal        21551
Premium      13791
Very Good    12082
Good          4906
Fair          1610
Name: count, dtype: int64

COLOR:
color
G    11292
E     9797
F     9542
H     8304
D     6775
I     5422
J     2808
Name: count, dtype: int64

CLARITY:
clarity
SI1     13065
VS2     12258
SI2      9194
VS1      8171
VVS2     5066
VVS1     3655
IF       1790
I1        741
Name: count, dtype: int64


In [29]:
# Estadísticas descriptivas
print("\n=== ESTADÍSTICAS DESCRIPTIVAS ===")
print(diamonds.describe())


=== ESTADÍSTICAS DESCRIPTIVAS ===
              carat         depth         table         price             x  \
count  53940.000000  53940.000000  53940.000000  53940.000000  53940.000000   
mean       0.797940     61.749405     57.457184   3932.799722      5.731157   
std        0.474011      1.432621      2.234491   3989.439738      1.121761   
min        0.200000     43.000000     43.000000    326.000000      0.000000   
25%        0.400000     61.000000     56.000000    950.000000      4.710000   
50%        0.700000     61.800000     57.000000   2401.000000      5.700000   
75%        1.040000     62.500000     59.000000   5324.250000      6.540000   
max        5.010000     79.000000     95.000000  18823.000000     10.740000   

                  y             z  
count  53940.000000  53940.000000  
mean       5.734526      3.538734  
std        1.142135      0.705699  
min        0.000000      0.000000  
25%        4.720000      2.910000  
50%        5.710000      3.530000  
7

## 3. Preparación de Datos y Entrenamiento del Modelo

### Variables seleccionadas:
- **Categóricas**: `cut` (calidad del corte) y `color` (color del diamante)
- **Numéricas**: `carat` (peso en quilates) y `depth` (profundidad total)
- **Variable objetivo**: `price` (precio)

In [30]:
# Seleccionar las variables para el modelo
# 2 categóricas: cut, color
# 2 numéricas: carat, depth
# Variable objetivo: price

# Crear una copia del dataset con las variables seleccionadas
df_model = diamonds[['cut', 'color', 'carat', 'depth', 'price']].copy()

print("Dataset para el modelo:")
print(df_model.head())
print(f"\nDimensiones: {df_model.shape}")

Dataset para el modelo:
       cut color  carat  depth  price
0    Ideal     E   0.23   61.5    326
1  Premium     E   0.21   59.8    326
2     Good     E   0.23   56.9    327
3  Premium     I   0.29   62.4    334
4     Good     J   0.31   63.3    335

Dimensiones: (53940, 5)


In [31]:
# Codificar variables categóricas
le_cut = LabelEncoder()
le_color = LabelEncoder()

df_model['cut_encoded'] = le_cut.fit_transform(df_model['cut'])
df_model['color_encoded'] = le_color.fit_transform(df_model['color'])

# Guardar las categorías originales para referencia
print("\nCodificación de CUT:")
for i, cat in enumerate(le_cut.classes_):
    print(f"{cat}: {i}")

print("\nCodificación de COLOR:")
for i, cat in enumerate(le_color.classes_):
    print(f"{cat}: {i}")


Codificación de CUT:
Fair: 0
Good: 1
Ideal: 2
Premium: 3
Very Good: 4

Codificación de COLOR:
D: 0
E: 1
F: 2
G: 3
H: 4
I: 5
J: 6


In [32]:
# Preparar X (features) y y (target)
X = df_model[['cut_encoded', 'color_encoded', 'carat', 'depth']]
y = df_model['price']

# Dividir en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Entrenar el modelo Random Forest
print("\n=== ENTRENAMIENTO DEL MODELO ===")
print("Entrenando Random Forest Regressor...")

model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)




=== ENTRENAMIENTO DEL MODELO ===
Entrenando Random Forest Regressor...


,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [33]:
# Evaluar el modelo
y_pred = model.predict(X_test)

print("\n=== EVALUACIÓN DEL MODELO ===")
print(f"R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"Mean Absolute Error: ${mean_absolute_error(y_test, y_pred):.2f}")
print(f"Root Mean Squared Error: ${np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")

# Importancia de las características
feature_importance = pd.DataFrame({
    'feature': ['cut', 'color', 'carat', 'depth'],
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n=== IMPORTANCIA DE LAS CARACTERÍSTICAS ===")
print(feature_importance)


=== EVALUACIÓN DEL MODELO ===
R² Score: 0.9062
Mean Absolute Error: $663.09
Root Mean Squared Error: $1221.34

=== IMPORTANCIA DE LAS CARACTERÍSTICAS ===
  feature  importance
2   carat    0.910250
3   depth    0.042969
1   color    0.034286
0     cut    0.012495


## 4. Sistema de Predicción Interactivo

In [34]:
def predecir_precio_diamante():
    """
    Función interactiva para predecir el precio de un diamante
    basado en las características ingresadas por el usuario.
    """
    print("\n" + "="*60)
    print("SISTEMA DE PREDICCIÓN DE PRECIOS DE DIAMANTES")
    print("="*60)
    
    # Solicitar CUT (calidad del corte)
    print("\n1. CALIDAD DEL CORTE (cut):")
    print("   Opciones disponibles:")
    for i, cat in enumerate(le_cut.classes_):
        print(f"   - {cat}")
    
    while True:
        cut_input = input("\n   Ingrese la calidad del corte: ").strip()
        if cut_input in le_cut.classes_:
            cut_encoded = le_cut.transform([cut_input])[0]
            break
        else:
            print(f"   ❌ Opción inválida. Por favor elija una de las opciones disponibles.")
    
    # Solicitar COLOR
    print("\n2. COLOR DEL DIAMANTE (color):")
    print("   Opciones disponibles (D=mejor, J=peor):")
    for i, cat in enumerate(le_color.classes_):
        print(f"   - {cat}")
    
    while True:
        color_input = input("\n   Ingrese el color: ").strip().upper()
        if color_input in le_color.classes_:
            color_encoded = le_color.transform([color_input])[0]
            break
        else:
            print(f"   ❌ Opción inválida. Por favor elija una de las opciones disponibles.")
    
    # Solicitar CARAT (peso)
    print("\n3. PESO EN QUILATES (carat):")
    print(f"   Rango típico: {diamonds['carat'].min():.2f} - {diamonds['carat'].max():.2f}")
    
    while True:
        try:
            carat_input = float(input("\n   Ingrese el peso en quilates: ").strip())
            if carat_input > 0:
                break
            else:
                print("   ❌ El peso debe ser un número positivo.")
        except ValueError:
            print("   ❌ Por favor ingrese un número válido.")
    
    # Solicitar DEPTH (profundidad)
    print("\n4. PROFUNDIDAD TOTAL (depth):")
    print(f"   Rango típico: {diamonds['depth'].min():.2f} - {diamonds['depth'].max():.2f}")
    
    while True:
        try:
            depth_input = float(input("\n   Ingrese la profundidad: ").strip())
            if depth_input > 0:
                break
            else:
                print("   ❌ La profundidad debe ser un número positivo.")
        except ValueError:
            print("   ❌ Por favor ingrese un número válido.")
    
    # Realizar la predicción
    input_data = np.array([[cut_encoded, color_encoded, carat_input, depth_input]])
    precio_estimado = model.predict(input_data)[0]
    
    # Mostrar resultados
    print("\n" + "="*60)
    print("RESULTADO DE LA PREDICCIÓN")
    print("="*60)
    print(f"\n📊 Características ingresadas:")
    print(f"   • Calidad del corte: {cut_input}")
    print(f"   • Color: {color_input}")
    print(f"   • Peso (quilates): {carat_input}")
    print(f"   • Profundidad: {depth_input}")
    print(f"\n💎 PRECIO ESTIMADO DEL DIAMANTE: ${precio_estimado:,.2f}")
    print("="*60 + "\n")
    
    return precio_estimado

# Ejecutar la función de predicción
precio = predecir_precio_diamante()


SISTEMA DE PREDICCIÓN DE PRECIOS DE DIAMANTES

1. CALIDAD DEL CORTE (cut):
   Opciones disponibles:
   - Fair
   - Good
   - Ideal
   - Premium
   - Very Good

2. COLOR DEL DIAMANTE (color):
   Opciones disponibles (D=mejor, J=peor):
   - D
   - E
   - F
   - G
   - H
   - I
   - J

3. PESO EN QUILATES (carat):
   Rango típico: 0.20 - 5.01

4. PROFUNDIDAD TOTAL (depth):
   Rango típico: 43.00 - 79.00

RESULTADO DE LA PREDICCIÓN

📊 Características ingresadas:
   • Calidad del corte: Good
   • Color: E
   • Peso (quilates): 3.0
   • Profundidad: 45.0

💎 PRECIO ESTIMADO DEL DIAMANTE: $11,039.18



## 5. Realizar Predicciones Adicionales

Ejecute la celda siguiente para realizar más predicciones:

In [35]:
# Ejecutar para hacer otra predicción
predecir_precio_diamante()


SISTEMA DE PREDICCIÓN DE PRECIOS DE DIAMANTES

1. CALIDAD DEL CORTE (cut):
   Opciones disponibles:
   - Fair
   - Good
   - Ideal
   - Premium
   - Very Good

2. COLOR DEL DIAMANTE (color):
   Opciones disponibles (D=mejor, J=peor):
   - D
   - E
   - F
   - G
   - H
   - I
   - J

3. PESO EN QUILATES (carat):
   Rango típico: 0.20 - 5.01

4. PROFUNDIDAD TOTAL (depth):
   Rango típico: 43.00 - 79.00

RESULTADO DE LA PREDICCIÓN

📊 Características ingresadas:
   • Calidad del corte: Premium
   • Color: H
   • Peso (quilates): 1.0
   • Profundidad: 55.0

💎 PRECIO ESTIMADO DEL DIAMANTE: $4,339.94



np.float64(4339.936761904762)